# CelebA Face Recognition with Triplet Loss

This notebook demonstrates how to train a Siamese Network with Triplet Loss on the CelebA Face Recognition Triplets dataset.

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from skimage import io
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image

# Check device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Data Preparation

We will use a dummy dataset generator for demonstration purposes. In a real scenario, you would download the CelebA dataset.

In [ ]:
def generate_dummy_data(base_dir, num_samples=100):
    """
    Generates dummy data simulating the CelebA Face Recognition Triplets structure.
    """
    data_dir = os.path.join(base_dir, 'images')
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
        
    csv_file = os.path.join(base_dir, 'train.csv')
    
    # Generate random images
    image_names = []
    for i in range(num_samples * 3): 
        img_name = f"img_{i:04d}.jpg"
        img_path = os.path.join(data_dir, img_name)
        
        # Create a random image (100x100 RGB)
        img_array = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        img.save(img_path)
        image_names.append(img_name)
        
    # Create triplets
    triplets = []
    for i in range(num_samples):
        anchor = image_names[i*3]
        positive = image_names[i*3+1]
        negative = image_names[i*3+2]
        triplets.append([anchor, positive, negative])
        
    df = pd.DataFrame(triplets, columns=['Anchor', 'Positive', 'Negative'])
    df.to_csv(csv_file, index=False)
    print(f"Generated {num_samples} dummy triplets in {base_dir}")

# Generate data
DATA_DIR = 'dummy_celeba_data'
if not os.path.exists(DATA_DIR):
    generate_dummy_data(DATA_DIR)

## 3. Dataset Class

In [ ]:
class APN_Dataset(Dataset):
    def __init__(self, df, data_dir):
        self.df = df
        self.data_dir = data_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        A_path = os.path.join(self.data_dir, row.Anchor)
        P_path = os.path.join(self.data_dir, row.Positive)
        N_path = os.path.join(self.data_dir, row.Negative)

        A_img = io.imread(A_path)
        P_img = io.imread(P_path)
        N_img = io.imread(N_path)
        
        A_img = torch.from_numpy(A_img).permute(2,0,1).float() / 255.0
        P_img = torch.from_numpy(P_img).permute(2,0,1).float() / 255.0
        N_img = torch.from_numpy(N_img).permute(2,0,1).float() / 255.0

        return A_img, P_img, N_img

## 4. Model Definition (EfficientNet-B0)

In [ ]:
class APN_Model(nn.Module):
    def __init__(self, emb_size=512):
        super(APN_Model, self).__init__()
        self.efficientnet = timm.create_model('efficientnet_b0', pretrained=True)
        self.efficientnet.classifier = nn.Linear(in_features=self.efficientnet.classifier.in_features, out_features=emb_size)

    def forward(self, images):
        embeddings = self.efficientnet(images)
        return embeddings

## 5. Training Loop

In [ ]:
def train_fn(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for A, P, N in tqdm(dataloader, desc="Training"):
        A, P, N = A.to(DEVICE), P.to(DEVICE), N.to(DEVICE)
        
        optimizer.zero_grad()
        
        A_embs = model(A)
        P_embs = model(P)
        N_embs = model(N)

        loss = criterion(A_embs, P_embs, N_embs)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    return total_loss / len(dataloader)

def eval_fn(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for A, P, N in tqdm(dataloader, desc="Validating"):
            A, P, N = A.to(DEVICE), P.to(DEVICE), N.to(DEVICE)
            A_embs = model(A)
            P_embs = model(P)
            N_embs = model(N)

            loss = criterion(A_embs, P_embs, N_embs)
            total_loss += loss.item()

    return total_loss / len(dataloader)

## 6. Execution

In [ ]:
# Configuration
BATCH_SIZE = 32
LR = 0.001
EPOCHS = 2 # Small number for demonstration
CSV_FILE = os.path.join(DATA_DIR, 'train.csv')
IMG_DIR = os.path.join(DATA_DIR, 'images')

# Load Data
df = pd.read_csv(CSV_FILE)
train_df, valid_df = train_test_split(df, test_size=0.20, random_state=42)

trainset = APN_Dataset(train_df, IMG_DIR)
validset = APN_Dataset(valid_df, IMG_DIR)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
validloader = DataLoader(validset, batch_size=BATCH_SIZE)

# Model Setup
model = APN_Model()
model.to(DEVICE)

criterion = nn.TripletMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Train
best_valid_loss = np.inf
for i in range(EPOCHS):
    print(f"Epoch {i+1}/{EPOCHS}")
    train_loss = train_fn(model, trainloader, optimizer, criterion)
    valid_loss = eval_fn(model, validloader, criterion)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Valid Loss: {valid_loss:.4f}")

    if valid_loss < best_valid_loss:
        torch.save(model.state_dict(), 'best_model.pt')
        best_valid_loss = valid_loss
        print("Saved Best Weights")

## 7. Evaluation

Load the best model and find nearest neighbors.

In [ ]:
def get_encoding(model, img_path):
    img = io.imread(img_path)
    img = torch.from_numpy(img).permute(2,0,1).float() / 255.0
    img = img.unsqueeze(0).to(DEVICE)
    
    model.eval()
    with torch.no_grad():
        enc = model(img)
    return enc.cpu().detach().numpy()

def euclidean_dist(img_enc, anc_enc_arr):
    dist = np.sqrt(np.sum((img_enc - anc_enc_arr)**2, axis=1))
    return dist

# Load best model
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

# Generate embeddings for database (Anchors)
database_encodings = []
database_paths = []

print("Generating embeddings...")
for index, row in df.iterrows():
    img_path = os.path.join(IMG_DIR, row['Anchor'])
    enc = get_encoding(model, img_path)
    database_encodings.append(enc)
    database_paths.append(img_path)

database_encodings = np.vstack(database_encodings)

# Test Query
query_idx = 0
query_img_name = df.iloc[query_idx]['Positive']
query_img_path = os.path.join(IMG_DIR, query_img_name)
print(f"Query Image: {query_img_path}")

query_enc = get_encoding(model, query_img_path)
distances = euclidean_dist(query_enc, database_encodings)
closest_indices = np.argsort(distances)[:5]

print("\nTop Matches:")
for i, idx in enumerate(closest_indices):
    print(f"{i+1}: {database_paths[idx]} (Distance: {distances[idx]:.4f})")